# Data Loading — formative2-hmm

Pipeline to ingest raw ZIP recordings from four activities — **standing, walking, jumping, still** — clean the sensor data, and write aligned CSVs to `data/processed/`.


## Setup — Imports, Constants & Paths

In [ ]:
import io, re, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# ── Constants ─────────────────────────────────────────────────────────────────
TARGET_HZ     = 100      # resample every recording to this rate (Hz)
EDGE_TRIM_SEC = 2.0      # seconds to remove from both ends of each recording
MERGE_TOL_SEC = 0.01     # acc ↔ gyro merge tolerance (~10 ms)
RANDOM_SEED   = 42
TRAIN_RATIO   = 0.8      # fraction of recordings assigned to training

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT    = Path("..").resolve()
RAW_DIR = ROOT / "data"
OUT_DIR = ROOT / "data" / "processed"

print(f"TARGET_HZ={TARGET_HZ}, EDGE_TRIM_SEC={EDGE_TRIM_SEC}, TRAIN_RATIO={TRAIN_RATIO}")
print(f"Root : {ROOT}")
print(f"Raw  : {RAW_DIR}")
print(f"Out  : {OUT_DIR}")

TARGET_HZ=100, EDGE_TRIM_SEC=2.0, TRAIN_RATIO=0.8


## 1 — Data Acquisition & Cleaning

### 1.1 — Inventory & naming convention
Raw recordings live under `data/<activity>/*.zip`, e.g. `data/walking/walking3.zip`.  
`_parse_name` extracts the activity label and recording id from each path; the train/test split is assigned **deterministically** via a hash of `(activity, rec_id)`.

In [ ]:
REC_ID_RE = re.compile(r"(\d+)\.zip$", re.IGNORECASE)

def _parse_name(zpath: Path):
    """Extract (split, activity, rec_id) from a path like data/walking/walking3.zip."""
    activity = zpath.parent.name.lower()
    m = REC_ID_RE.search(zpath.name)
    if not m:
        raise ValueError(f"Cannot parse recording id from: {zpath.name}")
    rec_id = int(m.group(1))
    h = hash((activity, rec_id, RANDOM_SEED)) & 0x7FFFFFFF
    split = "train" if (h / 0x7FFFFFFF) < TRAIN_RATIO else "test"
    return split, activity, rec_id

# ── Inventory ─────────────────────────────────────────────────────────────────
all_zips = sorted(RAW_DIR.glob("*/*.zip"))
print(f"Found {len(all_zips)} recordings:")
for zp in all_zips:
    split, act, rid = _parse_name(zp)
    print(f"  {act:10s}  rec={rid:>2d}  split={split}  {zp.name}")

Project root : /home/damour/Documents/ALU/formative2-hmm
Raw data dir : /home/damour/Documents/ALU/formative2-hmm/data
Output dir   : /home/damour/Documents/ALU/formative2-hmm/data/processed


### 1.2 — Unpack & standardize sensor files
For each ZIP:
1. Read the accelerometer and gyroscope CSVs  
2. Normalize the time axis to seconds-from-start and column names to `x/y/z`  
3. Drop duplicate timestamps and sort by time

In [ ]:
def _read_sensor(zp: zipfile.ZipFile, keywords: list):
    """Return first CSV matching keywords → DataFrame with [time_s, x, y, z]."""
    target = None
    for info in zp.infolist():
        nm = info.filename.lower()
        if nm.endswith(".csv") and any(k in nm for k in keywords):
            target = info; break
    if target is None:
        return None

    with zp.open(target) as f:
        df = pd.read_csv(io.BytesIO(f.read()))

    # pick time column
    lower = {c.lower(): c for c in df.columns}
    for cand in ("seconds_elapsed", "timestamp", "time", "epoch(s)", "epoch(ms)"):
        if cand in lower:
            tcol = lower[cand]; break
    else:
        tcol = df.columns[0]

    t = df[tcol].astype(float).to_numpy().copy()
    if t.max() > 1e6:
        t = t / 1000.0      # ms → s
    t = t - t[0]            # normalise to 0

    def pick(*names):
        for n in names:
            if n in df: return df[n].astype(float).to_numpy()
            if n.upper() in df: return df[n.upper()].astype(float).to_numpy()
        return None

    x = pick("x", "ax", "accx")
    y = pick("y", "ay", "accy")
    z = pick("z", "az", "accz")
    if x is None or y is None or z is None:
        others = [c for c in df.columns if c != tcol and pd.api.types.is_numeric_dtype(df[c])]
        if len(others) >= 3:
            x, y, z = (df[others[i]].to_numpy(float) for i in range(3))
        else:
            raise ValueError("x/y/z axes not found")

    out = pd.DataFrame({"time_s": t, "x": x, "y": y, "z": z})
    return out.drop_duplicates(subset="time_s").sort_values("time_s").reset_index(drop=True)

print("_read_sensor() defined ✓")

read_sensor() defined ✓


### 1.3 — Harmonize sampling rate
Phones may log at slightly different rates. We resample to a uniform **100 Hz** grid so windows and features are comparable across devices and recordings.

In [ ]:
def _estimate_hz(t: np.ndarray) -> float:
    """Estimate sampling rate (Hz) from a time array via median Δt."""
    if len(t) < 2: return float("nan")
    dt = np.diff(t); dt = dt[(dt > 0) & np.isfinite(dt)]
    return float(1.0 / np.median(dt)) if len(dt) else float("nan")


def _resample(df: pd.DataFrame, hz: int) -> pd.DataFrame:
    """Linear interpolation to a uniform *hz* Hz time grid."""
    if df is None or df.empty: return df
    t0, t1 = df["time_s"].iloc[0], df["time_s"].iloc[-1]
    new_t = np.arange(t0, t1 + 1e-9, 1.0 / hz)
    out = pd.DataFrame({"time_s": new_t})
    for c in ("x", "y", "z"):
        out[c] = np.interp(new_t, df["time_s"].to_numpy(), df[c].to_numpy())
    return out

print("_estimate_hz() and _resample() defined ✓")

estimate_hz() and resample() defined ✓


### 1.4 — Merge sensors & trim edges
We align accelerometer and gyroscope by nearest timestamp (≈10 ms tolerance), then trim **2.0 s** from both ends to remove start/stop noise added during recording.

In [ ]:
def _merge(acc, gyr, tol: float = MERGE_TOL_SEC) -> pd.DataFrame:
    """Nearest-time merge; returns [time_s, ax, ay, az, gx, gy, gz]."""
    COLS = ["time_s", "ax", "ay", "az", "gx", "gy", "gz"]
    if acc is None and gyr is None:
        return pd.DataFrame(columns=COLS)
    if acc is None:
        g = gyr.rename(columns={"x": "gx", "y": "gy", "z": "gz"}).copy()
        g[["ax", "ay", "az"]] = 0.0
        return g[COLS]
    if gyr is None:
        a = acc.rename(columns={"x": "ax", "y": "ay", "z": "az"}).copy()
        a[["gx", "gy", "gz"]] = 0.0
        return a[COLS]
    A = acc.sort_values("time_s")
    G = gyr.sort_values("time_s").rename(columns={"x": "gx", "y": "gy", "z": "gz"})
    m = pd.merge_asof(A, G, on="time_s", tolerance=tol, direction="nearest")
    m = m.dropna(subset=["gx", "gy", "gz"])
    return m.rename(columns={"x": "ax", "y": "ay", "z": "az"})[COLS].reset_index(drop=True)


def _trim(df: pd.DataFrame, sec: float = EDGE_TRIM_SEC) -> pd.DataFrame:
    """Remove the first and last *sec* seconds from a recording."""
    if df.empty: return df
    t0 = df["time_s"].iloc[0] + sec
    t1 = df["time_s"].iloc[-1] - sec
    if t1 <= t0: return df.iloc[0:0].copy()
    return df[(df["time_s"] >= t0) & (df["time_s"] <= t1)].reset_index(drop=True)

print("_merge() and _trim() defined ✓")

merge_sensors() defined ✓


### 1.5 — Save cleaned recordings
`unpack_and_clean_dir` orchestrates the full pipeline for every ZIP in `data/` and writes `*_cleaned.csv` files to `data/processed/train/` or `data/processed/test/`.

In [ ]:
def unpack_and_clean_dir(raw_dir: Path, out_dir: Path):
    """Process every ZIP under raw_dir/<activity>/ and write cleaned CSVs to out_dir."""
    (out_dir / "train").mkdir(parents=True, exist_ok=True)
    (out_dir / "test").mkdir(parents=True, exist_ok=True)

    for zpath in sorted(raw_dir.glob("*/*.zip")):
        split, activity, rec_id = _parse_name(zpath)
        with zipfile.ZipFile(zpath, "r") as zp:
            acc = _read_sensor(zp, ["acc", "accelerometer"])
            gyr = _read_sensor(zp, ["gyro", "gyroscope"])

        if acc is not None:
            hz = _estimate_hz(acc["time_s"].to_numpy())
            if np.isfinite(hz) and abs(hz - TARGET_HZ) > 1.0:
                acc = _resample(acc, TARGET_HZ)
        if gyr is not None:
            hz = _estimate_hz(gyr["time_s"].to_numpy())
            if np.isfinite(hz) and abs(hz - TARGET_HZ) > 1.0:
                gyr = _resample(gyr, TARGET_HZ)

        m = _merge(acc, gyr)
        m = _trim(m)
        m["activity"]     = activity
        m["split"]        = split
        m["recording_id"] = rec_id

        dest = out_dir / split / f"{zpath.stem}_cleaned.csv"
        m.to_csv(dest, index=False)

print("unpack_and_clean_dir() defined ✓")

trim_edges() and assign_split() defined ✓


### Run the pipeline

In [ ]:
print("Cleaning all recordings…")
unpack_and_clean_dir(RAW_DIR, OUT_DIR)

clean_train = sorted((OUT_DIR / "train").glob("*_cleaned.csv"))
clean_test  = sorted((OUT_DIR / "test").glob("*_cleaned.csv"))

print(f"\nCleaned CSVs → train: {len(clean_train)}, test: {len(clean_test)}")
print("Train sample:", [f.name for f in clean_train[:3]])
print("Test  sample:", [f.name for f in clean_test[:3]])

Found 50 recordings:
  jumping     rec= 3  2026-03-05_18-40-03.zip
  jumping     rec=35  2026-03-05_18-40-35.zip
  jumping     rec=54  2026-03-05_18-40-54.zip
  jumping     rec=13  2026-03-05_18-41-13.zip
  jumping     rec=31  2026-03-05_18-41-31.zip
  jumping     rec= 4  2026-03-05_18-42-04.zip
  jumping     rec= 8  2026-03-05_18-43-08.zip
  jumping     rec=27  2026-03-05_18-43-27.zip
  jumping     rec=47  2026-03-05_18-43-47.zip
  jumping     rec= 4  2026-03-05_18-44-04.zip
  jumping     rec=12  2026-03-05_19-05-12.zip
  jumping     rec=31  2026-03-05_19-05-31.zip
  standing    rec= 1  standing1.zip
  standing    rec=10  standing10.zip
  standing    rec=11  standing11.zip
  standing    rec=12  standing12.zip
  standing    rec= 2  standing2.zip
  standing    rec= 3  standing3.zip
  standing    rec= 4  standing4.zip
  standing    rec= 5  standing5.zip
  standing    rec= 6  standing6.zip
  standing    rec= 7  standing7.zip
  standing    rec= 8  standing8.zip
  standing    rec= 9  standi

## 2 — Verification

### 2.1 — Sampling rate check
Verify that cleaned files are uniformly sampled at ~100 Hz.

In [ ]:
def _estimate_hz_csv(csv_path: Path) -> float:
    t = pd.read_csv(csv_path, usecols=["time_s"])["time_s"].to_numpy()
    if len(t) < 2: return float("nan")
    dt = t[1:] - t[:-1]; dt = dt[dt > 0]
    return float(1.0 / pd.Series(dt).median()) if len(dt) else float("nan")

samples = (clean_train[:2] + clean_test[:2])[:4]
for p in samples:
    hz = _estimate_hz_csv(p)
    print(f"{p.name:55s}  ~{hz:.1f} Hz  (target={TARGET_HZ})")

  ✓  data/processed/train/2026-03-05_18-40-03_cleaned.csv  (321 rows)
  ✓  data/processed/test/2026-03-05_18-40-35_cleaned.csv  (331 rows)
  ✓  data/processed/train/2026-03-05_18-40-54_cleaned.csv  (325 rows)
  ✓  data/processed/train/2026-03-05_18-41-13_cleaned.csv  (324 rows)
  ✓  data/processed/train/2026-03-05_18-41-31_cleaned.csv  (304 rows)
  ✓  data/processed/test/2026-03-05_18-42-04_cleaned.csv  (323 rows)
  ✓  data/processed/train/2026-03-05_18-43-08_cleaned.csv  (343 rows)
  ✓  data/processed/train/2026-03-05_18-43-27_cleaned.csv  (318 rows)
  ✓  data/processed/train/2026-03-05_18-43-47_cleaned.csv  (342 rows)
  ✓  data/processed/test/2026-03-05_18-44-04_cleaned.csv  (311 rows)
  ✓  data/processed/train/2026-03-05_19-05-12_cleaned.csv  (374 rows)
  ✓  data/processed/train/2026-03-05_19-05-31_cleaned.csv  (341 rows)
  ✓  data/processed/train/standing1_cleaned.csv  (363 rows)
  ✓  data/processed/test/standing10_cleaned.csv  (344 rows)
  ✓  data/processed/train/standing11_cleane

### 2.2 — Schema & edge trim check
Confirm output columns and that the edge trim left the expected time window.

In [ ]:
if clean_train:
    df = pd.read_csv(clean_train[0])
    print(f"File   : {clean_train[0].name}")
    print(f"Shape  : {df.shape}")
    print(f"Columns: {list(df.columns)}")
    t0, t1 = float(df["time_s"].iloc[0]), float(df["time_s"].iloc[-1])
    print(f"time_s range: [{t0:.3f}, {t1:.3f}] s  (first ≈ {EDGE_TRIM_SEC} s after raw start)")
    print("Accel+Gyro columns present:", all(c in df.columns for c in ["ax","ay","az","gx","gy","gz"]))

=== Recordings per activity & split ===
split     test  train
activity             
jumping      3      9
standing     3      9
still        5      8
walking      2     11

=== Total rows per activity ===
activity
jumping     3957
standing    4170
still       4285
walking     4609
Name: rows, dtype: int64


In [ ]:
# ── Dataset summary ───────────────────────────────────────────────────────────
rows = []
for p in clean_train + clean_test:
    meta = pd.read_csv(p, usecols=["activity", "split", "recording_id"], nrows=1)
    rows.append({**meta.iloc[0].to_dict(), "n_rows": sum(1 for _ in open(p)) - 1})

summary = pd.DataFrame(rows)
print(f"Cleaned CSVs → train: {len(clean_train)}, test: {len(clean_test)}")
print("\n=== Recordings per activity & split ===")
print(summary.groupby(["activity", "split"]).size().unstack(fill_value=0))

if clean_train:
    print(f"\nSample file: {clean_train[0].name}")
    display(pd.read_csv(clean_train[0]).head(8))

Sample file : walking3_cleaned.csv
Shape       : (356, 10)
Columns     : ['time_s', 'ax', 'ay', 'az', 'gx', 'gy', 'gz', 'activity', 'split', 'recording_id']


,time_s,ax,ay,az,gx,gy,gz,activity,split,recording_id
0,2.007491,-0.096160,-0.464916,-1.275585,0.107068,0.070762,0.070313,walking,train,3
1,2.027591,-0.337566,-0.468229,-1.251572,0.221081,-0.044749,0.067032,walking,train,3
2,2.047641,-0.357419,-0.600447,-0.736800,0.197149,-0.136462,0.050162,walking,train,3
3,2.067742,-0.112711,-0.525914,-0.082661,0.065417,-0.181394,0.049579,walking,train,3
4,2.087817,0.204625,-0.473762,0.476483,0.028861,-0.052460,0.062369,walking,train,3
